
# Skadi BI - Gold

Fase inicial: lê Silver e grava as bases finais para os dashboards de Chamados e Superadmin.
Não leva CPF, senha, e-mail ou descrição de chamado para Gold.

In [ ]:
from pyspark.sql import DataFrame, functions as F

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

dbutils.widgets.text("nome_carga_rpa", "")
NOME_CARGA_RPA = dbutils.widgets.get("nome_carga_rpa").strip()

if not NOME_CARGA_RPA:
    raise ValueError("Informe o nome da carga RPA cadastrada em tb_controle_rpa.")

PERFIS_OFICIAIS = ["OPERADOR", "GESTOR", "ADMIN", "SUPER_ADMIN"]


def gravar_gold(df: DataFrame, nome_tabela: str) -> None:
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome_tabela)
    )

In [ ]:
chamados = spark.table("silver_solicitacao_suporte")
usuarios = spark.table("silver_usuario")
controle_rpa = spark.table("silver_controle_rpa")

gold_chamados = chamados.select(
    F.col("id").alias("chamado_id"),
    "cod_usuario_solicitante",
    "cod_usuario_responsavel",
    "categoria",
    "status",
    "data_hora_abertura",
    "data_hora_encerramento",
    F.col("status").isin("REGISTRADO", "EM_ATENDIMENTO").alias("esta_aberto"),
    (F.col("status") == "ATENDIDO").alias("esta_finalizado"),
    F.when(
        F.col("status") == "ATENDIDO",
        (F.unix_timestamp("data_hora_encerramento") - F.unix_timestamp("data_hora_abertura"))
        / F.lit(60.0),
    )
    .cast("double")
    .alias("tempo_resolucao_minutos"),
    F.current_timestamp().alias("atualizado_em"),
)

por_perfil = usuarios.groupBy("nivel_acesso").agg(
    F.count("id").alias("quantidade_usuarios")
)
perfis = spark.createDataFrame(
    [(perfil,) for perfil in PERFIS_OFICIAIS], ["nivel_acesso"]
)
total_usuarios = usuarios.agg(F.count("id").alias("total_usuarios"))

gold_superadmin_usuarios = (
    perfis.join(por_perfil, "nivel_acesso", "left")
    .crossJoin(total_usuarios)
    .fillna({"quantidade_usuarios": 0})
    .withColumn("atualizado_em", F.current_timestamp())
)

carga_rpa = spark.createDataFrame([(NOME_CARGA_RPA,)], ["nome_carga"])
gold_superadmin_rpa = (
    carga_rpa.join(controle_rpa, "nome_carga", "left")
    .select("nome_carga", "ultima_execucao_sucesso")
    .withColumn("atualizado_em", F.current_timestamp())
)

gravar_gold(gold_chamados, "gold_chamados")
gravar_gold(gold_superadmin_usuarios, "gold_superadmin_usuarios")
gravar_gold(gold_superadmin_rpa, "gold_superadmin_rpa")

In [ ]:
display(spark.table("gold_chamados"))
display(spark.table("gold_superadmin_usuarios"))
display(spark.table("gold_superadmin_rpa"))